"In this notebook, I tackled a very difficult problem: 3D bin packing. This problem is one of the most challenging in the combinatorial domain. As the size of the problem increases, reaching the optimal solution becomes quite difficult. In this problem, any box can rotate along the x, y, and z axes. However, by adjusting the constraints, you can not allow these rotations."

https://yetanothermathprogrammingconsultant.blogspot.com/2017/07/rectangles-no-overlap-constraints.html
https://yetanothermathprogrammingconsultant.blogspot.com/2021/10/2d-knapsack-problem.html#more

In [51]:
import gurobipy as gb
import numpy as np
import matplotlib.pyplot as plt

In [52]:
demand = [1,3,1,1,2,1,1,1,1,1,1] #demand quantity of every single box
number_of_demand = sum(demand)
width = [18.5,20.7,29.5,16,17.6,29,30.5,22.5,19.5,18.3,18.3]
length = [30,34.5,29.5,30.6,31.3,31.7,46.6,34.5,32,29.2,30.5]
height = [10,12,9.8,10.4,11.2,9.8,13.1,12,11.8,10,10.1]
container_width = 46
container_len = 80.8
container_height = 36.8 
M1 = container_width
M2 = container_len
M3 = container_height
print(sum(demand))
widt = {} #a dict for box widths
sayac = 0
sayı = 0
for j in demand:
    for i in range(j):
        widt[sayı] = width[sayac]
        sayı += 1
    sayac += 1
leng = {}#a dict for box lengths
sayac = 0
sayı = 0
for j in demand:
    for i in range(j):
        leng[sayı] = length[sayac]
        sayı += 1
    sayac += 1
binary = [(i,j,k) for i in range(sum(demand)) for j in range(sum(demand)) for k in range(4) if i < j and j-i != number_of_demand]
heig = {}#a dict for box heights
sayac = 0
sayı = 0
for j in demand:
    for i in range(j):
        heig[sayı] = height[sayac]
        sayı += 1
    sayac += 1
volume = {}
sayac = 0
sayı = 0
for j in demand:
    for i in range(j):
        volume[sayı] = width[sayac]*length[sayac]*height[sayac]
        sayı += 1
    sayac += 1
A = [(i,j) for i in range(sum(demand)) for j in range(sum(demand))  if i != j]
binaryy = [(i,j,k) for i in range(sum(demand)) for j in range(sum(demand)) for k in range(4) if i != j]

14


In [53]:
sum(volume.values())

112506.687

In [54]:
M1*M2*M3

136778.24

In [55]:
A = [(i,j,k) for i in range(sum(demand)) for j in ["width","length","height"] for k in ["width","length","height"]]

In [56]:
mdl = gb.Model("3dBinPacking")

In [57]:
x_low = mdl.addVars(range(sum(demand)), name = "x_low", ub = container_width)#starting point of the box along rhe x axis
y_low = mdl.addVars(range(sum(demand)),  name = "y_low", ub = container_len)#starting point of the box along rhe y axis
z_low = mdl.addVars(range(sum(demand)),  name = "z_low", ub = container_height)#starting point of the box along rhe z axis
ovrlp = mdl.addVars(binary, name = "overlap",vtype = gb.GRB.BINARY)#not overlap along the horizontal axis
ovrlp_z = mdl.addVars(binary, name = "overlap_yuks",vtype = gb.GRB.BINARY)#not overlap along the vertical axis
v = mdl.addVars(range(sum(demand)), name = "take",vtype = gb.GRB.BINARY)#select or not select a box
r = mdl.addVars(A, name = "rotation",vtype = gb.GRB.BINARY)#rotate or not rotate

In [58]:
env0 = mdl.getMultiobjEnv(0)
env1 = mdl.getMultiobjEnv(1)
env2 = mdl.getMultiobjEnv(2)

env0.setParam('TimeLimit', 600)
env1.setParam('TimeLimit', 60)
env2.setParam('TimeLimit', 60)

mdl.setObjectiveN(gb.quicksum(v[i]*volume[i] for i in range(sum(demand))), 0,2,1)#maximize number of box in the container 
mdl.setObjectiveN(gb.quicksum(z_low[i] for i in range(sum(demand))), 1,1,-1)#make sure no box is left hanging in the container
mdl.setObjectiveN(gb.quicksum(x_low[i]+y_low[i] for i in range(sum(demand))), 2,0,-1)#Box placements should be as compact as possible 
mdl.ModelSense = gb.GRB.MAXIMIZE

In [59]:
mdl.addConstr(gb.quicksum(v[i]*volume[i] for i in range(sum(demand))) <= M1*M2*M3 )#The total volume of the boxes inside the container cannot exceed the volume of the container.

<gurobi.Constr *Awaiting Model Update*>

In [60]:
mdl.addConstrs((z_low[i] + r[i,"height","height"]*heig[i]+r[i,"width","height"]*widt[i]+r[i,"length","height"]*leng[i] <= 
                z_low[j] + M3*(1-ovrlp_z[i,j,k]) 
    +M3*(1-v[i])+M3*(1-v[j]) for i,j,k in binary 
                if k == 0), name = "kısıt")#The boxes inside the container cannot overlap vertically

{(0, 1, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 5, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 6, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 7, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 8, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 9, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 10, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 11, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 12, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 13, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 2, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 3, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 7, 0): <gurobi.Constr *Awaiting Model

In [61]:
mdl.addConstrs((z_low[j] +  r[j,"height","height"]*heig[j]+r[j,"width","height"]*widt[j]+r[j,"length","height"]*leng[j]
                <= z_low[i] + M3*(1-ovrlp_z[i,j,k]) 
                +M3*(1-v[i])+M3*(1-v[j]) for i,j,k in binary
                 if k == 1), name = "kısıt1")#The boxes inside the container cannot overlap vertically

{(0, 1, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 5, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 6, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 7, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 8, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 9, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 10, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 11, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 12, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 13, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 2, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 3, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 7, 1): <gurobi.Constr *Awaiting Model

In [62]:
mdl.addConstrs((x_low[i] + r[i,"height","width"]*heig[i]+r[i,"width","width"]*widt[i]+r[i,"length","width"]*leng[i] <= 
                x_low[j] + M1*ovrlp[i,j,k] 
                + max(M1,M2,M3)*ovrlp_z[i,j,0]+ max(M1,M2,M3)*ovrlp_z[i,j,1]
+max(M1,M2,M3)*(1-v[i])+max(M1,M2,M3)*(1-v[j]) for i,j,k in binary
                if k == 0), name = "kısıt4")#The boxes inside the container cannot overlap horizontaly

{(0, 1, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 5, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 6, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 7, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 8, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 9, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 10, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 11, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 12, 0): <gurobi.Constr *Awaiting Model Update*>,
 (0, 13, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 2, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 3, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 0): <gurobi.Constr *Awaiting Model Update*>,
 (1, 7, 0): <gurobi.Constr *Awaiting Model

In [63]:
mdl.addConstrs((x_low[j] + r[j,"height","width"]*heig[j]+r[j,"width","width"]*widt[j]+r[j,"length","width"]*leng[j] <= 
                x_low[i] + M1*ovrlp[i,j,k]
                + max(M1,M2,M3)*ovrlp_z[i,j,0]+ max(M1,M2,M3)*ovrlp_z[i,j,1]
+max(M1,M2,M3)*(1-v[i])+max(M1,M2,M3)*(1-v[j]) for i,j,k in binary
                 if k == 1), name = "kısıt5")#The boxes inside the container cannot overlap horizontaly

{(0, 1, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 5, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 6, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 7, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 8, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 9, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 10, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 11, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 12, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 13, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 2, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 3, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 7, 1): <gurobi.Constr *Awaiting Model

In [64]:
mdl.addConstrs((y_low[i] + r[i,"height","length"]*heig[i]+r[i,"width","length"]*widt[i]+r[i,"length","length"]*leng[i] <= 
                y_low[j] + M2*ovrlp[i,j,k]
                + max(M1,M2,M3)*ovrlp_z[i,j,0]+ max(M1,M2,M3)*ovrlp_z[i,j,1]
+max(M1,M2,M3)*(1-v[i])+max(M1,M2,M3)*(1-v[j]) for i,j,k in binary
                if k == 2), name = "kısıt6")#The boxes inside the container cannot overlap horizontaly

{(0, 1, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 5, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 6, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 7, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 8, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 9, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 10, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 11, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 12, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 13, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 2, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 3, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 7, 2): <gurobi.Constr *Awaiting Model

In [65]:
mdl.addConstrs((y_low[j] + r[j,"height","length"]*heig[j]+r[j,"width","length"]*widt[j]+r[j,"length","length"]*leng[j] <= 
                y_low[i] + M2*ovrlp[i,j,k] 
                + max(M1,M2,M3)*ovrlp_z[i,j,0]+ max(M1,M2,M3)*ovrlp_z[i,j,1]
+max(M1,M2,M3)*(1-v[i])+max(M1,M2,M3)*(1-v[j]) for i,j,k in binary 
                 if k == 3), name = "kısıt7")#The boxes inside the container cannot overlap horizontaly

{(0, 1, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 5, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 6, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 7, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 8, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 9, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 10, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 11, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 12, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 13, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 2, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 3, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 7, 3): <gurobi.Constr *Awaiting Model

In [66]:
mdl.addConstrs((gb.quicksum(ovrlp[i,j,k] for k in range(4)) <= 3
for i in range(sum(demand)) for j in range(sum(demand)) if i < j and j-i != number_of_demand),name = "kısıt8")#The boxes inside the container cannot overlap horizontaly

{(0, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4): <gurobi.Constr *Awaiting Model Update*>,
 (0, 5): <gurobi.Constr *Awaiting Model Update*>,
 (0, 6): <gurobi.Constr *Awaiting Model Update*>,
 (0, 7): <gurobi.Constr *Awaiting Model Update*>,
 (0, 8): <gurobi.Constr *Awaiting Model Update*>,
 (0, 9): <gurobi.Constr *Awaiting Model Update*>,
 (0, 10): <gurobi.Constr *Awaiting Model Update*>,
 (0, 11): <gurobi.Constr *Awaiting Model Update*>,
 (0, 12): <gurobi.Constr *Awaiting Model Update*>,
 (0, 13): <gurobi.Constr *Awaiting Model Update*>,
 (1, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6): <gurobi.Constr *Awaiting Model Update*>,
 (1, 7): <gurobi.Constr *Awaiting Model Update*>,
 (1, 8): <gurobi.Constr *Awaiting Model Update

In [67]:
mdl.addConstrs((y_low[i] + r[i,"height","length"]*heig[i]+r[i,"width","length"]*widt[i]+r[i,"length","length"]*leng[i] <=
                container_len 
                for i in range(sum(demand)) )
                , name = "kısıt9")#The total length of the boxes along the y-axis cannot exceed the length of the container

{0: <gurobi.Constr *Awaiting Model Update*>,
 1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>,
 9: <gurobi.Constr *Awaiting Model Update*>,
 10: <gurobi.Constr *Awaiting Model Update*>,
 11: <gurobi.Constr *Awaiting Model Update*>,
 12: <gurobi.Constr *Awaiting Model Update*>,
 13: <gurobi.Constr *Awaiting Model Update*>}

In [68]:
mdl.addConstrs((x_low[i] +r[i,"height","width"]*heig[i]+r[i,"width","width"]*widt[i]+r[i,"length","width"]*leng[i] <= container_width  
                for i in range(sum(demand)) )
                , name = "kısıt10")#The total width of the boxes along the x-axis cannot exceed the width of the container

{0: <gurobi.Constr *Awaiting Model Update*>,
 1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>,
 9: <gurobi.Constr *Awaiting Model Update*>,
 10: <gurobi.Constr *Awaiting Model Update*>,
 11: <gurobi.Constr *Awaiting Model Update*>,
 12: <gurobi.Constr *Awaiting Model Update*>,
 13: <gurobi.Constr *Awaiting Model Update*>}

In [69]:
mdl.addConstrs((z_low[i] + r[i,"height","height"]*heig[i]+r[i,"width","height"]*widt[i]+r[i,"length","height"]*leng[i] <= 
                container_height  
                for i in range(sum(demand)) )
                , name = "kısıt11")#The total height of the boxes along the y-axis cannot exceed the height of the container

{0: <gurobi.Constr *Awaiting Model Update*>,
 1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>,
 9: <gurobi.Constr *Awaiting Model Update*>,
 10: <gurobi.Constr *Awaiting Model Update*>,
 11: <gurobi.Constr *Awaiting Model Update*>,
 12: <gurobi.Constr *Awaiting Model Update*>,
 13: <gurobi.Constr *Awaiting Model Update*>}

In [70]:
mdl.addConstrs((gb.quicksum(ovrlp_z[i,j,k] for k in range(4)) == 1
for i in range(sum(demand)) for j in range(sum(demand)) if i < j and j-i != number_of_demand ),name = "kısıt12")#The boxes inside the container cannot overlap vertically

{(0, 1): <gurobi.Constr *Awaiting Model Update*>,
 (0, 2): <gurobi.Constr *Awaiting Model Update*>,
 (0, 3): <gurobi.Constr *Awaiting Model Update*>,
 (0, 4): <gurobi.Constr *Awaiting Model Update*>,
 (0, 5): <gurobi.Constr *Awaiting Model Update*>,
 (0, 6): <gurobi.Constr *Awaiting Model Update*>,
 (0, 7): <gurobi.Constr *Awaiting Model Update*>,
 (0, 8): <gurobi.Constr *Awaiting Model Update*>,
 (0, 9): <gurobi.Constr *Awaiting Model Update*>,
 (0, 10): <gurobi.Constr *Awaiting Model Update*>,
 (0, 11): <gurobi.Constr *Awaiting Model Update*>,
 (0, 12): <gurobi.Constr *Awaiting Model Update*>,
 (0, 13): <gurobi.Constr *Awaiting Model Update*>,
 (1, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6): <gurobi.Constr *Awaiting Model Update*>,
 (1, 7): <gurobi.Constr *Awaiting Model Update*>,
 (1, 8): <gurobi.Constr *Awaiting Model Update

In [71]:
#symetry elemination constraint
for i in range(len(demand)):
    mdl.addConstrs(v[i] >= v[i+1] for i in range(sum(demand[:i+1])-demand[i],sum(demand[:i+1])-1))
    mdl.addConstrs(x_low[i] >= x_low[i+1] for i in range(sum(demand[:i+1])-demand[i],sum(demand[:i+1])-1))
    #mdl.addConstrs(y_low[i] >= y_low[i+1] for i in range(sum(talep[:i+1])-talep[i],sum(talep[:i+1])-1))
    #mdl.addConstrs(z_low[i] >= z_low[i+1] for i in range(sum(talep[:i+1])-talep[i],sum(talep[:i+1])-1))

mdl.addConstrs((x_low[i] <= v[i]*M1 for i in range(sum(talep))),name ="ososo")
mdl.addConstrs((ovrlp[i,j,k] <= v[i] for i,j,k in binary),name ="osososo")
mdl.addConstrs((ovrlp[i,j,k] <= v[j] for i,j,k in binary),name ="osososo_j")
#mdl.addConstrs((r[i] <= v[i] for i in range(sum(talep))),name ="osososo_r")
#mdl.addConstrs((y_low[i] <= v[i]*M2 for i in range(sum(talep))),name ="ososoy")
#mdl.addConstrs((z_low[i] <= v[i]*M3 for i in range(sum(talep))),name ="ososoz")

In [72]:
#mdl.addConstrs((x_low[i] <= v[i]*M1 for i in range(sum(talep))),name ="ososo")
#mdl.addConstrs((y_low[i] <= v[i]*M2 for i in range(sum(talep))),name ="ososo")
mdl.addConstrs((z_low[i] <= v[i]*M3 for i in range(sum(demand))),name ="ososo")

{0: <gurobi.Constr *Awaiting Model Update*>,
 1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>,
 9: <gurobi.Constr *Awaiting Model Update*>,
 10: <gurobi.Constr *Awaiting Model Update*>,
 11: <gurobi.Constr *Awaiting Model Update*>,
 12: <gurobi.Constr *Awaiting Model Update*>,
 13: <gurobi.Constr *Awaiting Model Update*>}

In [73]:
#A box can rotate either around the y-axis or around the z-axis
mdl.addConstrs(gb.quicksum(r[i,m,n] for n in ["width","length","height"]) == 1 for i in range(sum(demand))
              for m in ["width","length","height"])
mdl.addConstrs(gb.quicksum(r[i,m,n] for m in ["width","length","height"]) == 1 for i in range(sum(demand))
              for n in ["width","length","height"])

{(0, 'width'): <gurobi.Constr *Awaiting Model Update*>,
 (0, 'length'): <gurobi.Constr *Awaiting Model Update*>,
 (0, 'height'): <gurobi.Constr *Awaiting Model Update*>,
 (1, 'width'): <gurobi.Constr *Awaiting Model Update*>,
 (1, 'length'): <gurobi.Constr *Awaiting Model Update*>,
 (1, 'height'): <gurobi.Constr *Awaiting Model Update*>,
 (2, 'width'): <gurobi.Constr *Awaiting Model Update*>,
 (2, 'length'): <gurobi.Constr *Awaiting Model Update*>,
 (2, 'height'): <gurobi.Constr *Awaiting Model Update*>,
 (3, 'width'): <gurobi.Constr *Awaiting Model Update*>,
 (3, 'length'): <gurobi.Constr *Awaiting Model Update*>,
 (3, 'height'): <gurobi.Constr *Awaiting Model Update*>,
 (4, 'width'): <gurobi.Constr *Awaiting Model Update*>,
 (4, 'length'): <gurobi.Constr *Awaiting Model Update*>,
 (4, 'height'): <gurobi.Constr *Awaiting Model Update*>,
 (5, 'width'): <gurobi.Constr *Awaiting Model Update*>,
 (5, 'length'): <gurobi.Constr *Awaiting Model Update*>,
 (5, 'height'): <gurobi.Constr *Await

In [75]:
#optional additional scenarios, a constraint could be that all boxes should touch the base of the container, 
for i in range(sum(demand)):
    z_low[i].setAttr("ub",0)

In [76]:
mdl.params.LiftProjectCuts = 2
#mdl.params.Heuristics = 0.7
#mdl.params.TimeLimit = 2000
#mdl.params.Presolve = 2
#mdl.params.MIPGap = 0.6
#mdl.params.MIPFocus = 2
mdl.params.Cuts = 3
mdl.optimize()

Set parameter LiftProjectCuts to value 2
Set parameter Cuts to value 3
Gurobi Optimizer version 10.0.0 build v10.0.0rc2 (win64)

CPU model: Intel(R) Core(TM) i7-3630QM CPU @ 2.40GHz, instruction set [SSE2|AVX]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 875 rows, 910 columns and 6298 nonzeros
Model fingerprint: 0xd4f7d24e
Variable types: 42 continuous, 868 integer (868 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [1e+00, 2e+04]
  Bounds range     [1e+00, 8e+01]
  RHS range        [1e+00, 1e+05]

---------------------------------------------------------------------------
Multi-objectives: starting optimization with 3 objectives ... 
---------------------------------------------------------------------------

Multi-objectives: applying initial presolve ...
---------------------------------------------------------------------------

Presolve removed 36 rows and 22 columns
Presolve time: 0.01s
Pr

     0     0  -90.00000    0  229 -643.60000  -90.00000  86.0%     -   26s
     0     2  -90.00000    0  200 -643.60000  -90.00000  86.0%     -   26s
H  154   165                    -641.0000000  -98.46599  84.6%   131   26s
H  225   240                    -638.1000000  -98.87214  84.5%   106   26s
H  273   286                    -635.1000000  -98.87214  84.4%  97.5   27s
H  294   304                    -634.0000000  -98.87214  84.4%  92.6   27s
  1072   919 -133.05841   32  252 -634.00000 -101.34534  84.0%  48.9   30s
H 1464  1124                    -599.0000000 -101.34534  83.1%  81.4   32s
H 1518  1116                    -597.3000000 -101.34534  83.0%  81.4   32s
H 1522  1073                    -591.6000000 -101.34534  82.9%  81.4   32s
  1683  1158 -304.22121   56   79 -591.60000 -101.34534  82.9%  80.7   35s
  3871  2353 -200.22463   30  120 -591.60000 -106.60064  82.0%  78.5   40s
  6185  3801 -145.72501   20  237 -591.60000 -112.23660  81.0%  78.4   47s
  7771  5289 -116.62753  

In [77]:
import plotly.graph_objects as go
colors= ["red","yellow","orange","grey","green","purple","blue","magenta","cyan","pink","darkseagreen","chocolate"
       ,"lightsalmon","aqua","coral","darkred","azure","limegreen","bisque","black"]
eslesme = []
toplam = demand[0]
for i in range(len(demand)):
    for j in range(toplam-demand[i],toplam):
        eslesme.append((i,j)) 
    if i < len(demand)-1:
        toplam += demand[i+1]
    else:
        break
        
# Kocaman küpü çiz
fig = go.Figure(data=[
    go.Mesh3d(
        x=[0,0,M1,M1,0,0,M1,M1],
        y=[0,M2,M2,0,0,M2,M2,0],
        z=[0,0,0,0,M3,M3,M3,M3],
        
        opacity=0.5,
        color='grey'
    )
]) 

kutu = []
ana_liste = []
for i in x_low:
    if v[i].x > 0:
        p1 = list([x_low[i].x,y_low[i].x,z_low[i].x])
        p2= list([x_low[i].x+r[i,"height","width"].x*heig[i]+r[i,"width","width"].x*widt[i]+r[i,"length","width"].x*leng[i],y_low[i].x,z_low[i].x])
        p3 = list([x_low[i].x+r[i,"height","width"].x*heig[i]+r[i,"width","width"].x*widt[i]+r[i,"length","width"].x*leng[i],y_low[i].x+r[i,"height","length"].x*heig[i]+r[i,"width","length"].x*widt[i]+r[i,"length","length"].x*leng[i],z_low[i].x])
        p4= list([x_low[i].x,y_low[i].x+r[i,"height","length"].x*heig[i]+r[i,"width","length"].x*widt[i]+r[i,"length","length"].x*leng[i],z_low[i].x])
        
        p5= list([x_low[i].x,y_low[i].x,z_low[i].x+r[i,"height","height"].x*heig[i]+r[i,"width","height"].x*widt[i]+r[i,"length","height"].x*leng[i]])
        p6= list([x_low[i].x+r[i,"height","width"].x*heig[i]+r[i,"width","width"].x*widt[i]+r[i,"length","width"].x*leng[i],y_low[i].x,z_low[i].x+r[i,"height","height"].x*heig[i]+r[i,"width","height"].x*widt[i]+r[i,"length","height"].x*leng[i]])
        p7= list([x_low[i].x+r[i,"height","width"].x*heig[i]+r[i,"width","width"].x*widt[i]+r[i,"length","width"].x*leng[i],y_low[i].x+r[i,"height","length"].x*heig[i]+r[i,"width","length"].x*widt[i]+r[i,"length","length"].x*leng[i],z_low[i].x+r[i,"height","height"].x*heig[i]+r[i,"width","height"].x*widt[i]+r[i,"length","height"].x*leng[i]])
        p8= list([x_low[i].x,y_low[i].x+r[i,"height","length"].x*heig[i]+r[i,"width","length"].x*widt[i]+r[i,"length","length"].x*leng[i],z_low[i].x+r[i,"height","height"].x*heig[i]+r[i,"width","height"].x*widt[i]+r[i,"length","height"].x*leng[i]])
        
        for h in eslesme:
            if h[1] == i:
                s= h[0]


        fig.add_trace(
            go.Mesh3d(
                # 8 vertices of a cube
                x=[p1[0],p2[0],p3[0],p4[0],p5[0],p6[0],p7[0],p8[0]],
                y=[p1[1],p2[1],p3[1],p4[1],p5[1],p6[1],p7[1],p8[1]],
                z=[p1[2],p2[2],p3[2],p4[2],p5[2],p6[2],p7[2],p8[2]],

                color=colors[s],
                colorscale=[[0, colors[s]],
                    [0.5, colors[s]],
                    [1, colors[s]]],
                opacity=1,
                intensity = np.linspace(0, 1, 8, endpoint=True),
                # i, j and k give the vertices of triangles
                i = [7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2],
                j = [3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3],
                k = [0, 7, 2, 3, 6, 7, 1, 1, 5, 5, 7, 6],

                name="K_"+str(s)+"_"+str(i),
                showscale=False
            ))
        fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0], p3[0], p4[0], p1[0], p5[0], p6[0], p7[0], p8[0], p5[0], p6[0], p2[0], p3[0], p7[0], p8[0], p4[0]],
                y=[p1[1], p2[1], p3[1], p4[1], p1[1], p5[1], p6[1], p7[1], p8[1], p5[1], p6[1], p2[1], p3[1], p7[1], p8[1], p4[1]],
                z=[p1[2], p2[2], p3[2], p4[2], p1[2], p5[2], p6[2], p7[2], p8[2], p5[2], p6[2], p2[2], p3[2], p7[2], p8[2], p4[2]],
                mode='lines',
                line=dict(color='black', width=2),
                name="K_"+str(s)+"_"+str(i),
                showlegend=True
            ))

fig.show()